In [2]:
# SCENARIO: “AI Banking Assistant with Role-Based Access”
# 🏦 Background Story

# A bank builds an AI assistant to help users:

# Check account balance
# View transactions
# Approve loans
# Manage customer accounts

# 👉 But not everyone can do everything

# 🧑‍🤝‍🧑 Roles in the Bank
# Role	Description
# 👤 Customer	Bank account holder
# 👨‍💼 Employee	Bank staff
# 🧑‍💼 Manager	Branch manager
# 🔐 Permissions (RBAC)
# Action	Customer	Employee	Manager
# View own balance	✅	✅	✅
# View others' accounts	❌	✅	✅
# Approve loan	❌	❌	✅
# View all transactions	❌	✅	✅


# ======================================
# STEP 1: Install Libraries
# ======================================
!pip install groq gradio


# ======================================
# STEP 2: Load API Key from Colab Secrets
# ======================================
# ======================================
# STEP 2: Load API Key using os.environ
# ======================================
import os

# 🔑 Set your API key here (only for Colab testing)
os.environ["GROQ_API_KEY"] = "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"

from groq import Groq
client = Groq(api_key=os.getenv("GROQ_API_KEY"))


# ======================================
# STEP 3: Dummy Bank Database
# ======================================
accounts = {
    "1001": {"name": "Amit", "balance": 50000},
    "1002": {"name": "Neha", "balance": 75000}
}


# ======================================
# STEP 4: Tool Functions (APIs)
# ======================================
def get_balance(account_id):
    if account_id in accounts:
        return f"💰 Balance of {account_id}: ₹{accounts[account_id]['balance']}"
    return "❌ Account not found"


def approve_loan(account_id):
    if account_id in accounts:
        return f"🏦 Loan approved for account {account_id}"
    return "❌ Account not found"


# ======================================
# STEP 5: RBAC Security Layer
# ======================================
def secure_access(role, user_account, requested_account, action):

    # Manager → full access
    if role == "manager":
        return True

    # Employee → can view all but cannot approve loans
    elif role == "employee":
        if action == "approve_loan":
            return False
        return True

    # Customer → only own account, no loan approval
    elif role == "customer":
        return user_account == requested_account and action != "approve_loan"

    return False


# ======================================
# STEP 6: MCP Tool Decision via LLM
# ======================================
def decide_action(query):
    try:
        prompt = f"""
        You are an AI banking assistant.

        Decide which action to take:
        - get_balance
        - approve_loan

        Rules:
        - Balance related queries → get_balance
        - Loan approval queries → approve_loan

        Return ONLY the action name.

        Query: {query}
        """

        response = client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=[{"role": "user", "content": prompt}]
        )

        action = response.choices[0].message.content.strip().lower()
        return action

    except Exception as e:
        print("❌ Groq Error:", e)
        return "fallback"


# ======================================
# STEP 7: MCP Agent (Core Logic)
# ======================================
def banking_agent(message, role, user_account, requested_account, history):

    # Input validation
    if not user_account:
        response = "⚠️ Please enter your account ID"
        history.append((message, response))
        return "", history

    if not requested_account:
        requested_account = user_account  # default to own account

    # Step 1: LLM decides action
    action = decide_action(message)

    # Step 2: RBAC Security Check
    if not secure_access(role, user_account, requested_account, action):
        response = "🚫 Access Denied: You are not authorized"

    else:
        # Step 3: Tool Invocation
        if "balance" in action:
            response = get_balance(requested_account)

        elif "loan" in action:
            response = approve_loan(requested_account)

        # Fallback if LLM fails
        elif action == "fallback":
            msg_lower = message.lower()
            if "balance" in msg_lower:
                response = get_balance(requested_account)
            elif "loan" in msg_lower:
                response = approve_loan(requested_account)
            else:
                response = "⚠️ Could not understand request"

        else:
            response = "🤖 Try asking about balance or loan"

    # Save chat history
    history.append((message, response))

    return "", history


# ======================================
# STEP 8: Gradio UI
# ======================================
import gradio as gr

with gr.Blocks() as demo:

    gr.Markdown("# 🏦 AI Banking Assistant (MCP + RBAC + Groq)")

    role = gr.Dropdown(
        ["customer", "employee", "manager"],
        label="Select Role"
    )

    user_account = gr.Textbox(label="Your Account ID (e.g., 1001)")
    requested_account = gr.Textbox(label="Target Account ID (optional)")

    chatbot = gr.Chatbot(height=400)
    msg = gr.Textbox(label="Ask your question")

    state = gr.State([])

    msg.submit(
        banking_agent,
        inputs=[msg, role, user_account, requested_account, state],
        outputs=[msg, chatbot]
    )


# ======================================
# STEP 9: Launch App
# ======================================
demo.launch(share=True)

/tmp/ipykernel_5189/338451582.py:192: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot(height=400)
/tmp/ipykernel_5189/338451582.py:192: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your chatbot.
  chatbot = gr.Chatbot(height=400)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://843dcc4c3ae068e690.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [3]:
# ======================================
# STEP 1: Install Libraries
# ======================================
!pip install groq nest_asyncio


# ======================================K
# STEP 2: Load Groq API Key (Colab Secret)
# ======================================
# ======================================
# STEP 2: Load API Key using os.environ
# ======================================
import os

# 🔑 Set your API key here (only for Colab testing)
os.environ["GROQ_API_KEY"] = "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"

from groq import Groq
client = Groq(api_key=os.getenv("GROQ_API_KEY"))


# ======================================
# STEP 3: MOCK TOOLS (Simulating MCP Tools)
# ======================================

import asyncio
import random
import nest_asyncio

# Apply nest_asyncio to allow nested event loops
nest_asyncio.apply()

async def web_search(query):
    await asyncio.sleep(1)  # simulate delay
    return f"📰 News about {query}: Market is growing fast."

async def get_stock_data(company):
    await asyncio.sleep(1)
    price = random.randint(100, 500)
    return f"📈 Stock price of {company}: ${price}"

async def fetch_company_profile(company):
    await asyncio.sleep(1)
    return f"👥 {company} has 5000 employees, HQ in USA"


# ======================================
# STEP 4: PARALLEL TOOL INVOCATION
# ======================================

async def parallel_research(company):

    results = await asyncio.gather(
        web_search(company),
        get_stock_data(company),
        fetch_company_profile(company),
        return_exceptions=True
    )

    news, stock, profile = results

    return {
        "news": news if not isinstance(news, Exception) else "News unavailable",
        "stock": stock if not isinstance(stock, Exception) else "Stock unavailable",
        "profile": profile if not isinstance(profile, Exception) else "Profile unavailable"
    }


# ======================================
# STEP 5: CHAINED TOOL INVOCATION USING GROQ
# ======================================

def analyse_text(text):
    response = client.chat.completions.create(
    model="llama-3.3-70b-versatile", # Updated model name to a currently available Groq model
        messages=[{
            "role": "user",
            "content": f"Analyze this data and give key insights:\n{text}"
        }]
    )
    return response.choices[0].message.content


def generate_report(analysis, company):
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile", # Updated model name to a currently available Groq model
        messages=[{
            "role": "user",
            "content": f"Create a professional report for {company}:\n{analysis}"
        }]
    )
    return response.choices[0].message.content


# ======================================
# STEP 6: FULL MCP PIPELINE
# ======================================

async def full_pipeline(company):

    # Step 1: Parallel Data Collection
    data = await parallel_research(company)

    combined_text = f"""
    News: {data['news']}
    Stock: {data['stock']}
    Profile: {data['profile']}
    """

    # Step 2: Analysis (LLM)
    analysis = analyse_text(combined_text)

    # Step 3: Report Generation (LLM)
    report = generate_report(analysis, company)

    return report


# ======================================
# STEP 7: RUN THE SYSTEM
# ======================================

company_name = "Tesla"

# Use asyncio.run() after applying nest_asyncio
result = asyncio.run(full_pipeline(company_name))

print("📊 FINAL REPORT:\n")
print(result)

📊 FINAL REPORT:

**Tesla, Inc. Market Analysis and Growth Prospects Report**

**Executive Summary**

This report provides an analysis of Tesla, Inc., a leading player in the electric vehicle and renewable energy sectors. Based on the available data, the report highlights the company's strong market presence, significant workforce, and potential for growth and expansion. With a growing market and a relatively stable stock price, Tesla presents an attractive investment opportunity for those interested in the electric vehicle and renewable energy sectors.

**Market Growth and Trends**

The market for electric vehicles and renewable energy solutions is experiencing rapid growth, driven by increasing demand for sustainable and environmentally friendly technologies. This trend is expected to continue, providing a positive indicator for Tesla's future prospects. The company's expertise in electric vehicles, solar energy, and energy storage solutions positions it well to capitalize on this gro